In [ ]:
import polars as pl
import polars.selectors as cs
import numpy as np
import scipy
import warnings
import ray
import os
import json
import itertools
import glob
import yaml
from tqdm import tqdm
from datetime import datetime, timedelta

with open("config.yaml") as f:
    config = yaml.safe_load(f)


MIMIC_PATH = config["mimic_path"]
SEPTIC_SCHOCK_PATH = config["output_path"]
META_DATA_PATH = config["meta_data_path"]
ERROR_DATE = datetime.fromisoformat('4444-04-04 04:00:00')

In [ ]:
def get_relevant_intersection(truth_df, intersect_field, target_path):
    other_df = pl.scan_csv(target_path)
    return truth_df.join(other_df, on=intersect_field, how='inner')

In [ ]:
rel_items_df = pl.scan_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_items.csv')

In [ ]:
fstay_df = pl.read_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_icu_stay.csv')

In [ ]:
chart_df = get_relevant_intersection(fstay_df, 'hadm_id', f'{MIMIC_PATH}/icu/chartevents.csv').collect()

In [ ]:
chart_df.write_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_chartevents.csv')

In [ ]:
get_relevant_intersection(fstay_df, 'hadm_id', f'{MIMIC_PATH}/hosp/labevents.csv').collect().write_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_labevents.csv')

In [ ]:
with open(f'{META_DATA_PATH}/covariate_mappings.json', 'r', encoding='utf8') as fp:
    variable_mappings = json.load(fp)

In [ ]:
df = pl.DataFrame(
    {
        "value": [99999, None, 69 ,123],
         "convertable": ["1", "22", "1231", "86"],
    }
)
df.filter(~pl.col('value').is_null()).sort(by='value').select(
    pl.col('*').exclude('convertable'),
    pl.col('convertable').cast(pl.Int32)
)

In [ ]:
def find_time_inner(bp_list: pl.Series, max_value: float) -> pl.Expr:
        pass
test = pl.scan_csv(f'{MIMIC_PATH}/icu/chartevents.csv').select(
    pl.col('valuenum'),
    pl.col('charttime').str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col('hadm_id'),
    pl.col('subject_id'),
    pl.col('itemid')
).head(100)
test.collect()

In [ ]:
def find_time(hadm_id, chart_df):
    def find_time_inner(bp_list: pl.Series, max_value: float)-> pl.Expr:
        try:
            return rel_measure.filter(
                (pl.col('itemid').is_in(bp_list)) & (pl.col('valuenum') < max_value)
            ).select(pl.col('charttime')).head(1).item()
        except ValueError:
            return ERROR_DATE

    sys_bp =  pl.Series([220050, 220179, 224167, 225309, 227243])
    dial_bp = pl.Series([220051, 220180, 224643, 225310, 227242])
    mean_bp = pl.Series([220052, 220181, 225312])
    all_bp = pl.concat([
        sys_bp,
        dial_bp,
        mean_bp
    ])
    rel_measure = chart_df.filter(pl.col('hadm_id') == hadm_id).filter(
        ~pl.col('valuenum').is_null()
    ).sort(by='charttime').filter(
        pl.col('itemid').is_in(all_bp)
    ).select(
        pl.col('itemid'), 
        pl.col('charttime'),
        pl.col('valuenum')
    ).collect()
    try:
        baseline = rel_measure.filter(
            pl.col('itemid').is_in(sys_bp)
        ).select(
            pl.col('valuenum')
        ).head(1).item()
    except ValueError:
        baseline = 0
    time_candidates = [
        find_time_inner(sys_bp, 90),
        find_time_inner(mean_bp, 65),
        find_time_inner(sys_bp, baseline - 40),
    ]
    return min(time_candidates)

In [ ]:
with open(f'{SEPTIC_SCHOCK_PATH}/covariant_icds.json', 'r', encoding='utf8') as fp:
    variable_mappings = json.load(fp)
variable_mappings

In [ ]:
chart_key = list()
chart_num = list()

lab_key = list()
lab_num = list()

for key, (num, df) in variable_mappings.items():
    if df == 'chart_df':
        chart_key.append(key)
        chart_num.append(int(num))
    else:
        lab_key.append(key)
        lab_num.append(int(num))
chart_var_map_df =  pl.LazyFrame({
    'variable_name': chart_key,
    'variable_num': chart_num
})
lab_var_map_df =  pl.LazyFrame({
    'variable_name': lab_key,
    'variable_num': lab_num
})

In [ ]:
def create_checklist(hadm_id, target_df, chart_df, variable_mappings, time_zero):
    return target_df.filter(
        (pl.col('charttime') > time_zero) & (pl.col('hadm_id') == hadm_id) & (pl.col('itemid').is_in(
            variable_mappings.select(
                pl.col('variable_num')
            ).collect()
        ))).join(variable_mappings, left_on='itemid', right_on='variable_num', how='left').group_by(
            'hadm_id', 'subject_id', 'variable_name'
        ).agg(pl.col('valuenum', 'charttime').implode()).collect()

In [ ]:
# load relevant info
fstay_df = pl.read_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_icu_stay.csv')
lab_df = pl.scan_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_labevents.csv')
chart_df = pl.scan_csv(f'{SEPTIC_SCHOCK_PATH}/relevant_chartevents.csv')

chart_df = chart_df.with_columns(
    pl.col('charttime').str.to_datetime("%Y-%m-%d %H:%M:%S")
)

lab_df = lab_df.with_columns(
    pl.col('charttime').str.to_datetime("%Y-%m-%d %H:%M:%S")
)

In [ ]:
os.path.isfile(f'{SEPTIC_SCHOCK_PATH}/casual_info.json')

In [ ]:
# load logic
if os.path.isfile(f'{SEPTIC_SCHOCK_PATH}/casual_info.json'):
    aggr_df = pl.read_json(f'{SEPTIC_SCHOCK_PATH}/casual_info.json')
    aggr_df = aggr_df.with_columns(
        pl.col('charttime').list.eval(pl.element().list.eval(pl.element().str.to_datetime("%Y-%m-%d %H:%M:%S")))
    )
    fstay_df = fstay_df.join(aggr_df, on='hadm_id', how='anti')
else:
    aggr_df = pl.DataFrame(schema = {
        'hadm_id' : pl.Int64,
        'subject_id' : pl.Int64,
        'variable_name' : str,
        'valuenum' : list[list[float]],
        'charttime' : list[list[datetime]]
    })

In [ ]:
save_timer = 500
current = 0
for (current_hadm_id,) in tqdm(fstay_df.select(pl.col('hadm_id')).iter_rows()):
    time_zero =  find_time(current_hadm_id, chart_df)
    current += 1
    if time_zero >= ERROR_DATE:
        #print('Skip')
        aggr_df = pl.concat([
            aggr_df, pl.DataFrame({
                'hadm_id' : current_hadm_id,
                'subject_id' : None,
                'variable_name' : None,
                'valuenum' : None,
                'charttime' : None
            }) 
        ])
        continue
    chart_checklist = create_checklist(current_hadm_id, chart_df, chart_df, chart_var_map_df, time_zero)

    lab_checklist = create_checklist(current_hadm_id, lab_df, chart_df, lab_var_map_df, time_zero)

    aggr_df = pl.concat([
        aggr_df, chart_checklist, lab_checklist
    ])
    if current > save_timer:
        aggr_df.write_json(f'{SEPTIC_SCHOCK_PATH}/casual_info.json')
        current = 0
aggr_df.write_json(f'{SEPTIC_SCHOCK_PATH}/casual_info.json')

In [ ]:
aggr_df.filter(pl.col('hadm_id') == 29079034)

In [ ]:
asd = pl.scan_csv(f'{MIMIC_PATH}/hosp/labevents.csv')
print(asd.head(10).collect())

In [ ]:
icu_df = pl.scan_csv(f'{MIMIC_PATH}/icu/icustays.csv').select(
    pl.col('subject_id'),
    pl.col('hadm_id'),
    pl.col('stay_id'),
    pl.col('intime').str.to_datetime("%Y-%m-%d %H:%M:%S"),
    pl.col('outtime').str.to_datetime("%Y-%m-%d %H:%M:%S")
).with_columns(
    pl.col('intime').dt.offset_by('1d').alias('time_limit'),
).with_columns(
    pl.when(
        pl.col('time_limit') > pl.col('outtime')).then(
            pl.col('outtime')
        ).otherwise(
            pl.col('time_limit')
        ).alias('time_limit')
).sort('intime').unique(
    subset=['hadm_id'], maintain_order=True, keep='first'
)

In [ ]:
pat_df = pl.scan_csv(f'{MIMIC_PATH}/hosp/patients.csv')
ourdata_df = pl.scan_csv(f'{SEPTIC_SCHOCK_PATH}/completefirst_input_with_pat_info.csv')
filterer_df = ourdata_df.join(on='subject_id', how='left', other=pat_df.select(
    pl.col('subject_id'),
    pl.col('dod').str.to_datetime("%Y-%m-%d")
))

In [ ]:
filterer_df = filterer_df.join(
    on='hadm_id',
    how='left',
    other=icu_df.select(pl.all().exclude('subject_id'))
).filter(
    (pl.col('dod').dt.offset_by('1d') >= pl.col('time_limit')) | (pl.col('dod').is_null())
)

In [ ]:
sofa_df = pl.scan_csv(f'{SEPTIC_SCHOCK_PATH}/sofa_scores.csv').with_columns(
    pl.col('starttime').str.to_datetime("%Y-%m-%dT%H:%M:%S"),
    pl.col('endtime').str.to_datetime("%Y-%m-%dT%H:%M:%S")
)
aps3_df = pl.scan_csv(f'{SEPTIC_SCHOCK_PATH}/apsiii_corrected_scores.csv')
aps3_df.unique(subset=['subject_id', 'hadm_id']).collect()

In [ ]:
filterer_df = filterer_df.join(
    on='stay_id',
    how='left',
    other=sofa_df
).sort('starttime').unique(
    subset=['stay_id'], maintain_order=True, keep='first'
)

In [ ]:
final_stuff = filterer_df.join(
    on='stay_id',
    how='left',
    other=aps3_df.select(
        pl.col('stay_id'),
        pl.col('apsiii')
    )
).collect()

In [ ]:
final_stuff.select(
    pl.all().exclude(
        'dod', 'intime', 'outtime', 'time_limit', 'starttime', 'endtime', 'stay_id'
    )
).write_csv(f'{SEPTIC_SCHOCK_PATH}/sofa_enchanced_data.csv')

In [ ]:
data_df = pl.scan_csv(f'{SEPTIC_SCHOCK_PATH}/sofa_enchanced_data.csv')
adm_df = pl.scan_csv(f'{MIMIC_PATH}/hosp/admissions.csv')
pat_df = pl.scan_csv(f'{MIMIC_PATH}/hosp/patients.csv')
icu_stay_df = pl.scan_csv(f'{MIMIC_PATH}/icu/icustays.csv')

In [ ]:
data_df = data_df.join(adm_df.select(
    pl.col('hadm_id'),
    pl.col('subject_id'),
    pl.col('hospital_expire_flag')
), on=['hadm_id', 'subject_id'], how='left')

In [ ]:
data_df = data_df.join(icu_stay_df.select(
    pl.col('hadm_id'),
    pl.col('subject_id'),
    pl.col('los'),
    pl.col('intime').str.to_datetime("%Y-%m-%d %H:%M:%S")
).sort('intime').unique(subset=['subject_id','hadm_id'], maintain_order=True, keep='first'), on=['hadm_id', 'subject_id'], how='left').with_columns(
    pl.col('los').alias('icu_los')
).drop('los', 'intime')

In [ ]:
data_df.join( pat_df.join(data_df.select(
        pl.col('subject_id'),
        pl.col('hadm_id')
    ), on='subject_id', how='inner').select(
        pl.col('dod').str.to_datetime("%Y-%m-%d"),
        pl.col('subject_id'),
        pl.col('hadm_id')
    ).join(icu_stay_df.select(
        pl.col('subject_id'),
        pl.col('hadm_id'),
        pl.col('outtime').str.to_datetime("%Y-%m-%d %H:%M:%S")
    ).sort('outtime').unique(subset=['hadm_id','subject_id'], maintain_order=True, keep='first'), on=['hadm_id', 'subject_id'], how='left').with_columns(
        (pl.col('dod') <= pl.col('outtime')).alias('icu_expire_flag')
    ).join(
        adm_df.select(
            pl.col('subject_id'),
            pl.col('hadm_id'),
            pl.col('admittime').str.to_datetime("%Y-%m-%d %H:%M:%S")
        ),
        on=['hadm_id', 'subject_id'],
        how='left'
    ).with_columns(
        (pl.col('admittime').dt.offset_by('1y') >= pl.col('dod')).fill_null(False).alias('one_year_mort'),
        (pl.col('admittime').dt.offset_by('1mo') >= pl.col('dod')).fill_null(False).alias('one_month_mort'),
        pl.col('icu_expire_flag').fill_null(False),
    ).select(
        pl.col('icu_expire_flag'),
        pl.col('one_year_mort'),
        pl.col('one_month_mort'),
        pl.col('subject_id'),
        pl.col('hadm_id')
    ), on=['subject_id', 'hadm_id'], how='left'
).collect().write_csv(f'{SEPTIC_SCHOCK_PATH}/additional_outcome_data_fixed.csv')

In [ ]:
pat_df.collect().join(all_data_df, on='subject_id', how='right').select(pl.col(
    'dod'
)).count()

In [ ]:
all_data_df = pl.read_csv(f'{SEPTIC_SCHOCK_PATH}/additional_outcome_data_fixed.csv')
all_data_df